# Notebook 03: tiny scalar backward engine

In notebook 02, we computed the chain rule by hand. In this notebook, we build a tiny teaching engine that stores the same forward values and backward sensitivities inside Python objects. The goal is not to replace the production MLP code; it is to make the idea behind `backward()` visible.

## Start with the manual expression

First, repeat the same expression from notebook 02. These plain floats give us a baseline: we know the forward values should be `d = -6.0`, `e = 4.0`, and `f = 16.0`. Later, our `Value` objects should produce the same forward numbers while also remembering the graph.

In [1]:
a = 2.0
b = -3.0
c = 10.0

d = a * b
e = d + c
f = e * e

print(d, e, f)

-6.0 4.0 16.0


## From hand bookkeeping to object bookkeeping

The manual backward pass worked because we kept track of each intermediate value and each local derivative. A `Value` object will keep that bookkeeping next to the number itself. For now, it stores the forward number in `data`, the backward sensitivity in `grad`, the parent values in `_prev`, and the operation name in `_op`.

## Add local backward rules to `Value`

The graph now needs more than parent links. Each operation also stores a small `_backward` function on its output. That function knows one local chain-rule step: addition passes the output gradient unchanged to both parents, while multiplication sends the output gradient scaled by the other parent's forward value.

In [2]:
from typing import Callable


class Value:
    def __init__(
        self,
        data: float,
        _children: tuple["Value", ...] = (),
        _op: str = "",
    ) -> None:
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward: Callable[[], None] | None = lambda: None

    def __repr__(self) -> str:
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other: "Value") -> "Value":
        out = Value(data=self.data + other.data, _children=(self, other), _op="+")

        def _backward() -> None:
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other: "Value") -> "Value":
        out = Value(data=self.data * other.data, _children=(self, other), _op="*")

        def _backward() -> None:
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

## Leaf values

A leaf value is an input we create directly, like `a = Value(2.0)`. It has a number in `data`, starts with `grad = 0.0`, and has no parents because no earlier operation created it.

In [3]:
a = Value(2.0)
b = Value(-3.0)

print(a)
print(b)

Value(data=2.0, grad=0.0)
Value(data=-3.0, grad=0.0)


## Addition creates a graph node

When we write `c = a + b`, Python calls `a.__add__(b)`. The result `c` stores the forward value `-1.0`, remembers that the operation was `+`, and keeps links back to both parent objects. Those links are what the backward pass will use later to update `a.grad` and `b.grad`.

In [4]:
a = Value(2.0)
b = Value(-3.0)
c = a + b

print(c)
print(c._op)
print(c._prev)

Value(data=-1.0, grad=0.0)
+
{Value(data=-3.0, grad=0.0), Value(data=2.0, grad=0.0)}


## Inspect the parent links

The parent links store the actual `Value` objects, not just the numbers `2.0` and `-3.0`. Printing only each parent's `.data` is a gentle way to check which forward values are connected to `c`. Because `_prev` is a set, the order may change, but both parents should be present.

In [5]:
print(c.data)
print([parent.data for parent in c._prev])

-1.0
[-3.0, 2.0]


## Rebuild the expression with `Value` objects

Now use the same expression shape as the plain-float example, but stop at `e = d + c`. The forward numbers should still match notebook 02. The difference is that `d` and `e` now also carry parent links and local backward rules.

In [6]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

d = a * b
e = d + c

print(d)
print(e)
print(e._op)
print([parent.data for parent in e._prev])

Value(data=-6.0, grad=0.0)
Value(data=4.0, grad=0.0)
+
[10.0, -6.0]


## Finish the forward graph

The final step is `f = e * e`, so the same `Value` object is used twice. The printed parent list only shows one `4.0` because `_prev` is a set, and a set keeps one copy of the same object. During backward, this repeated use is still important: `e` should receive gradient from both sides of the multiplication.

In [7]:
f = e * e

print(f)
print(f._op)
print([parent.data for parent in f._prev])

Value(data=16.0, grad=0.0)
*
[4.0]


## Test one local backward rule

Before automating the whole graph, test one operation by hand. In this tiny test, we pretend `c` is the final output. A value's gradient means "how much the chosen final output changes when this value changes."

That is why we set `c.grad = 1.0`: if `c` is the chosen final output, then `dc/dc = 1.0`. In plain words, `c` changes one-for-one with itself. Calling `c._backward()` then applies only the local addition rule, so both `a.grad` and `b.grad` should increase by `1.0`.

In [8]:
a = Value(2.0)
b = Value(-3.0)
c = a + b

c.grad = 1.0
c._backward()

print(a.grad)
print(b.grad)

1.0
1.0
